In [ ]:
import sys
import pytz
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPOSITORY_DIR = NOTEBOOK_DIR.parents[3]
if str(REPOSITORY_DIR) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_DIR))
from Data_query.trino_config import trino_parallel_batch
from LSO_anti_islanding.conformance.solar_analytics_workflow.trino.trino_connection_on_ec2 import (
    engine,
    iceberg_exec,
    iceberg_sql,
)
# from visualisation import *

In [ ]:
num_workers = 1

In [ ]:
# stop_trino() 

In [ ]:
# sleep(120)

In [ ]:
# sleep(180)

In [ ]:
# big_workers = 1
# workers = 0
# num_workers = max(workers, big_workers)
# ensure_trino_running(worker_desired_count=workers, big_worker_desired_count=big_workers)
# sleep(60)

In [ ]:
iceberg_exec("DROP TABLE IF EXISTS all_uncurtailedPV_LSO")
iceberg_exec("""CREATE TABLE all_uncurtailedPV_LSO (
                site_id BIGINT,
                t_stamp TIMESTAMP,
                year INT,
                month INT,
                uncurtailed_P DOUBLE,
                P_kw DOUBLE,
                GHI DOUBLE,
                n_train BIGINT
            )
             WITH (
                format = 'PARQUET',
                partitioning = ARRAY['year', 'month'],
                sorted_by = ARRAY['site_id', 't_stamp']
             )
    """)

In [ ]:
df0 = iceberg_sql("""select  a.site_id, max(uncurtailed_P - S_99) as max_diff, max(S_99) as S_99, max(ac_capacity_kw) as ac_capacity_kw
            from  all_uncurtailedPV_LSO a left join (select distinct site_id, S_99, ac_capacity_kw from meta_up23c) as m on a.site_id = m.site_id
            where uncurtailed_P - S_99 > .5
            group by a.site_id
            order by max_diff desc""")

In [ ]:
df0

In [ ]:
df1 = iceberg_sql("""select  a.site_id, max(uncurtailed_P - S_99) as max_diff, max(S_99) as S_99, max(ac_capacity_kw) as ac_capacity_kw
            from  all_uncurtailedPV_LSO a left join (select distinct site_id, S_99, ac_capacity_kw from meta_up23c) as m on a.site_id = m.site_id
            where uncurtailed_P - S_99 > .5
            group by a.site_id
            order by max_diff desc""")

In [ ]:
df1

In [ ]:
iceberg_sql("""select  *
            from  all_uncurtailedPV_LSO a left join (select distinct site_id, S_99, ac_capacity_kw from meta_up23c) as m on a.site_id = m.site_id
            where a.site_id = 146075438 
            and uncurtailed_p > 200
            limit 10""")

In [ ]:
iceberg_sql("""select  * 
            from pv_ghi_norm_model
            where site_id=1792599725
            order by abs(a) desc""")

In [ ]:
iceberg_sql("""select  * 
            from split_days
            where site_id = 146075438
            and day_type = 'train'
            limit 100""")

In [ ]:
sleep(20)

In [ ]:
acceptible_sites = (
    iceberg_sql("""
        SELECT DISTINCT site_id
        FROM iceberg.solar_analytics_iceberg.lso_anti_islanding_conformance
        WHERE assessment_status <> 'unassessed'
            AND site_id IS NOT NULL
        ORDER BY site_id
    """)
    .get_column("site_id")
    .to_list()
)
acceptible_sites = ", ".join(map(str, acceptible_sites))

In [ ]:
num_parts = 3
time_bin_interval = "5"  # in minutes
model = "pv_ghi_norm_model"
# removed this filter to include assessed sites
# the num cohort is already small
# and i cant find the csv anywehre

# acceptible_sites = ", ".join(
#     map(str, pd.read_csv("mape<50_sites.csv")["site_id"].tolist())
# )


def run_func(args):
    year, month, part = args
    # time_filter = f"year = {year} and month = {month}"
    time_filter = f"year = {year}"
    part_filter = f"site_id % {num_parts} = {part}"
    df = iceberg_exec(f"""
                    insert into all_uncurtailedPV_LSO
                    with eligible_data AS (
                        SELECT
                            site_id,
                            actual_day,
                            t_stamp,
                            CAST(
                                date_trunc('minute', t_stamp + interval '10' hour)
                                - interval '1' minute * (minute(t_stamp + interval '10' hour) % {time_bin_interval})
                                AS TIME) AS tod_bin,
                            GHI/GHI_cs AS x,
                            GHI_cs,
                            P_kw_norm/ NULLIF(P_kw_norm_cs, 0.0) AS y,
                                P_kw_norm,
                                P_kw_norm_cs,
                                S_norm,
                                S_99,
                                V
                        FROM structured_data
                        WHERE P_kw_norm_cs > 0.2 AND GHI > 50 and P_kw_norm >= 0 and P_kw_norm <= P_kw_norm_cs
                            and {time_filter} and {part_filter} and site_id in ({acceptible_sites})
                    ),
                    validation_on_eligible_data AS (
                        select 
                            t.site_id, 
                            t.t_stamp, 
                            x as GHI,
                            P_kw_norm, 
                            case when P_kw_norm_cs * (a + b * x) >= P_kw_norm then P_kw_norm_cs * (a + b * x) else P_kw_norm end AS P_kw_norm_est,
                            V,
                            S_norm,
                            S_99, 
                            m.n as n_train
                        from eligible_data t 
                            join {model} m on t.site_id = m.site_id and t.tod_bin = m.tod_bin
                    )
                    SELECT site_id, t_stamp, year(t_stamp) as year, month(t_stamp) as month, P_kw_norm_est*S_99 as uncurtailed_P, P_kw_norm*S_99, GHI, n_train 
                    FROM validation_on_eligible_data
                        where P_kw_norm_est is not null
                        """)

    #

    # sleep(10)
    print(f"Completed {time_filter},  part {part}")
    return df


tasks = [
    (year, month, part)
    for year in (2024, 2025)
    for month in range(1, 2)
    for part in range(0, num_parts)
]
#   for split_cons in ['system.bucket(postcode, 16) > -1'] ]

try:
    df = trino_parallel_batch(
        run_func, tasks, num_workers=num_workers, batch_size=num_workers
    )
except Exception as e:
    print(f"Error during data retrieval: {e}")
finally:
    # stop_trino()
    pass
# df['t_stamp'] = pd.to_datetime(df['t_stamp']).dt.tz_localize('utc').dt.tz_convert(pytz.FixedOffset(10*60))
df

In [ ]:
iceberg_sql("select * from all_uncurtailedPV_LSO where uncurtailed_P < P_kw limit 10")